# Gravity Lens — Starter Notebook

In this task, you are given simulated images of gravitationally lensed sources. Each image is a 64×64 grayscale map. The lensing is modeled by a 2D Gaussian kernel whose width parameters are determined by:

- **Mass** (*M*) – a continuous value between 1.0 and 10.0
- **Ellipticity** (*e*) – a binary label: 0 for spherical, 1 for elliptical.

You must build a model that, given an image, predicts both the mass (regression) and the ellipticity class (classification).

**Subtask A (50 points)**: Predict mass (RMSE).
**Subtask B (50 points)**: Predict ellipticity (accuracy).

This starter notebook provides a baseline solution that you can improve upon. Run top to bottom to generate data, train a simple model, and create a submission file.

**You only need to implement the `predict_A` and `predict_B` methods in the `Submission` class at the bottom.**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import cloudpickle
import os
from scipy.ndimage import gaussian_filter

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

### 1. Generate Training and Test Data
In this notebook we generate the data on the fly. In a real competition the data would be provided as `.npy` files. We include a generation function so you can test locally.

In [ ]:
def generate_image(mass, ellip, size=64, source_seed=None):
    """Generate a 64x64 image from a random source and lensing kernel."""
    if source_seed is not None:
        np.random.seed(source_seed)
    # Random source: uniform noise plus a few bright spots
    source = np.random.rand(size, size) * 0.5
    for _ in range(5):
        x, y = np.random.randint(0, size, 2)
        source[x, y] += np.random.rand() * 2.0
    source = np.clip(source, 0, 1)

    # Lens kernel widths
    sig_x = mass * (1 + ellip)
    sig_y = mass * (1 - ellip)
    # Convolve with Gaussian
    image = gaussian_filter(source, sigma=(sig_y, sig_x), mode='constant', cval=0.0)
    # Add noise
    noise = np.random.normal(0, 0.02, (size, size))
    image = np.clip(image + noise, 0, 1)
    return image.astype(np.float32)

def generate_dataset(n_samples, save=False):
    X = np.zeros((n_samples, 1, 64, 64), dtype=np.float32)
    y_mass = np.zeros(n_samples, dtype=np.float32)
    y_ellip = np.zeros(n_samples, dtype=np.int64)
    for i in range(n_samples):
        mass = np.random.uniform(1.0, 10.0)
        ellip = np.random.randint(0, 2)
        X[i, 0] = generate_image(mass, ellip)
        y_mass[i] = mass
        y_ellip[i] = ellip
    return X, y_mass, y_ellip

# Generate training and test sets (test without labels)
X_train, y_mass, y_ellip = generate_dataset(1000)
X_test, _, _ = generate_dataset(300)  # test labels are hidden

# Save for later use (optional)
np.save('train_images.npy', X_train)
np.save('train_labels_mass.npy', y_mass)
np.save('train_labels_ellip.npy', y_ellip)
np.save('test_images.npy', X_test)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

### 2. Define a Baseline Model
This is a simple CNN with two output heads. You can improve it by changing architecture, adding regularization, or using data augmentation.

In [ ]:
class GravityLensCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
        )
        # After three poolings: 64 -> 32 -> 16 -> 8 => feature size: 64 * 8 * 8 = 4096
        self.fc = nn.Linear(64 * 8 * 8, 128)
        self.fc_mass = nn.Linear(128, 1)
        self.fc_ellip = nn.Linear(128, 2)

    def forward(self, x):
        x = self.conv(x)
        x = torch.relu(self.fc(x))
        mass = self.fc_mass(x)
        ellip = self.fc_ellip(x)
        return mass, ellip

# Instantiate model
model = GravityLensCNN()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

### 3. Training Loop
The baseline trains for 50 epochs. You may adjust epochs, learning rate, batch size, or add a learning rate scheduler.

In [ ]:
# Prepare data loaders
X_t = torch.from_numpy(X_train).float()
y_mass_t = torch.from_numpy(y_mass).float().view(-1, 1)
y_ellip_t = torch.from_numpy(y_ellip).long()

# Train/val split
N = len(X_t)
split = int(0.8 * N)
indices = torch.randperm(N)
train_idx, val_idx = indices[:split], indices[split:]
train_dataset = TensorDataset(X_t[train_idx], y_mass_t[train_idx], y_ellip_t[train_idx])
val_dataset = TensorDataset(X_t[val_idx], y_mass_t[val_idx], y_ellip_t[val_idx])

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Losses and optimizer
criterion_mass = nn.MSELoss()
criterion_ellip = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 50
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        images, mass, ellip = batch
        images, mass, ellip = images.to(device), mass.to(device), ellip.to(device)
        optimizer.zero_grad()
        pred_mass, pred_ellip = model(images)
        loss = criterion_mass(pred_mass, mass) + criterion_ellip(pred_ellip, ellip)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        val_acc_ellip = 0.0
        for batch in val_loader:
            images, mass, ellip = batch
            images, mass, ellip = images.to(device), mass.to(device), ellip.to(device)
            pred_mass, pred_ellip = model(images)
            loss = criterion_mass(pred_mass, mass) + criterion_ellip(pred_ellip, ellip)
            val_loss += loss.item()
            pred_ellip_class = pred_ellip.argmax(1)
            val_acc_ellip += (pred_ellip_class == ellip).float().mean().item()
        val_loss /= len(val_loader)
        val_acc_ellip /= len(val_loader)
    print(f"Epoch {epoch+1:2d}: Train Loss {total_loss:.4f}, Val Loss {val_loss:.4f}, Val Acc (ellip) {val_acc_ellip:.4f}")

# Save the trained model for later use
torch.save(model.state_dict(), 'gravity_lens_model.pth')

### 4. Submission Class
The grader will load this class and call `predict_A` and `predict_B`. You can replace the model or add any preprocessing inside these methods.

In [ ]:
class Submission:
    def __init__(self):
        # Load the trained model
        self.model = GravityLensCNN()
        self.model.load_state_dict(torch.load('gravity_lens_model.pth', map_location='cpu'))
        self.model.eval()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

    def predict_A(self, images):
        # images: np array of shape (N, 1, 64, 64)
        with torch.no_grad():
            images_t = torch.from_numpy(images).float().to(self.device)
            mass, _ = self.model(images_t)
            return mass.cpu().numpy().flatten()

    def predict_B(self, images):
        with torch.no_grad():
            images_t = torch.from_numpy(images).float().to(self.device)
            _, ellip = self.model(images_t)
            return ellip.argmax(1).cpu().numpy()

# Create the submission object
sol = Submission()

# Optionally test on a few images
print("Predictions on first 5 test images:")
print("Mass:", sol.predict_A(X_test[:5]))
print("Ellipticity:", sol.predict_B(X_test[:5]))

### 5. Build submission.pkl
This file is what you upload to the judge.

In [ ]:
with open('submission.pkl', 'wb') as f:
    cloudpickle.dump(sol, f)
mb = os.path.getsize('submission.pkl') / 1e6
print(f"submission.pkl created ({mb:.2f} MB) – must be < 50 MB")
assert mb < 50, "File too large!"

### 6. (Optional) Local Evaluation
If you have ground truth for the test set (not provided in the real task), you can evaluate your performance.

In [ ]:
# Simulate evaluation (requires test labels which are hidden in competition)
# For local testing, we'll use the test set we generated (but in reality these would be hidden)
# Load test labels (we have them because we generated the data)
try:
    y_test_mass = np.load('test_labels_mass.npy')
    y_test_ellip = np.load('test_labels_ellip.npy')
    pred_mass = sol.predict_A(X_test)
    pred_ellip = sol.predict_B(X_test)
    from sklearn.metrics import mean_squared_error, accuracy_score
    rmse = np.sqrt(mean_squared_error(y_test_mass, pred_mass))
    acc = accuracy_score(y_test_ellip, pred_ellip)
    print(f"RMSE (mass): {rmse:.4f}")
    print(f"Accuracy (ellip): {acc:.4f}")
except FileNotFoundError:
    print("Test labels not found; skip local evaluation.")

### 7. Submit
Upload `submission.pkl` to the judge.